# PCA Analysis: Dopamine & GABA Dynamics during Stimulus-Reward Learning

Config-driven PCA dimensionality reduction across datasets (SpontFB, CRFB, ToneFB) and neuron populations (Dopamine, GABA, Combined).

**Outputs**: PNG figures, manifest JSONs, and a summary CSV under `outputs/`.

In [3]:
import os
import pandas as pd
import kaleido

#custom built plotting and analysis functions
from plot_pca import plot_pca, run_analysis

## Configuration

Edit the config dict below to control which datasets, neuron combinations, and parameters are used.

In [9]:
config = {
    # ---- Dataset definitions ----
    "datasets": {
        "SpontFB": {"mat_file": "dataSpontFB.mat", "var_name": "dataSpontFB"},
        "CRFB":    {"mat_file": "dataCRFB.mat",    "var_name": "dataCRFB"},
        "ToneFB":  {"mat_file": "dataToneFB.mat",  "var_name": "dataToneFB"},
    },

    # ---- Neuron combination definitions ----
    "neuron_combos": {
        "Dopamine": ["DF", "DB", "D", "DFB"], ######## QUESTION: D AND DF IN DOPAMINE OR JUST SELECTIVELY TUNED NEURONS?'
        "GABA":     ["GF", "GB", "G", "GFB"], ######## Same question for GABA
        "Combined": ["DF", "DB", "GF", "GB", "D", "G", "DFB", "GFB"], #### and here too...
    },

    # ---- PCA parameters ----
    "pca": {"n_components": 3},

    # ---- Windowing parameters ----
    "window": {
        "event_idx": 600,   # index of the event within each half ### a bit rustic, but event is always at 600
        "window": 120,      # +/- timesteps around event (>=100 for ToneFB Reward)
        "dt": 0.01,         # seconds per timestep
    },

    # ---- Smoothing parameters ----
    "smoothing": {"sg_window": 11, "sg_order": 3},

    # ---- Visualization ----
    "visualization": {
        "fwd_cmap": "YlOrRd",
        "bwd_cmap": "Blues",
        "fig_width": 1100,
        "fig_height": 850,
        "plot_types": ["trajectory"], # or + 'scatter'
    },

    # ---- Output ----
    "output": {
        "base_dir": "outputs",
        "save_png": True,
        "save_manifest": True,
        "save_summary_csv": True,
        "show_figures": False,   # set True to display interactive plots
    },
}

## Run Full Analysis Pipeline

Iterates all dataset x neuron-combo combinations, generates figures, saves PNGs + manifests, and produces a summary CSV.

In [10]:
summary = run_analysis(config)
print(f"Completed. Summary CSV: {summary['summary_csv_path']}")

Completed. Summary CSV: outputs\analysis_summary.csv


## Summary Table

Displays all generated files and explained-variance metrics.

In [11]:
summary_df = pd.DataFrame(summary['results'])
display(summary_df[[
    'dataset', 'neuron_combo', 'neuron_groups', 'n_neurons',
    'explained_var_pc1', 'explained_var_pc2', 'explained_var_pc3',
    'explained_var_total', 'status', 'warnings', 'error'
]])

,dataset,neuron_combo,neuron_groups,n_neurons,explained_var_pc1,explained_var_pc2,explained_var_pc3,explained_var_total,status,warnings,error
0,SpontFB,Dopamine,DF+DB+D+DFB,730.0,0.069875,0.028825,0.013543,0.112244,success,PNG export failed for SpontFB_Dopamine_scatter...,NaN
1,SpontFB,GABA,GF+GB+G+GFB,NaN,NaN,NaN,NaN,NaN,error,,Unable to allocate 306. MiB for an array with ...
2,SpontFB,Combined,DF+DB+GF+GB+D+G+DFB+GFB,NaN,NaN,NaN,NaN,NaN,error,,Unable to allocate 306. MiB for an array with ...
3,CRFB,Dopamine,DF+DB+D+DFB,NaN,NaN,NaN,NaN,NaN,error,,Input X contains NaN.\nPCA does not accept mis...
4,CRFB,GABA,GF+GB+G+GFB,155.0,0.270364,0.104279,0.041766,0.416410,success,,NaN
5,CRFB,Combined,DF+DB+GF+GB+D+G+DFB+GFB,NaN,NaN,NaN,NaN,NaN,error,,Input X contains NaN.\nPCA does not accept mis...
6,ToneFB,Dopamine,DF+DB+D+DFB,NaN,NaN,NaN,NaN,NaN,error,,Input X contains NaN.\nPCA does not accept mis...
7,ToneFB,GABA,GF+GB+G+GFB,155.0,0.298261,0.116999,0.045324,0.460585,success,,NaN
8,ToneFB,Combined,DF+DB+GF+GB+D+G+DFB+GFB,NaN,NaN,NaN,NaN,NaN,error,,Input X contains NaN.\nPCA does not accept mis...


## Inspect Individual Results

Display a specific figure interactively by selecting dataset and neuron combo.

In [8]:
# Choose which result to inspect
dataset_key = "SpontFB"
combo_key = "Dopamine"

key = f"{dataset_key}_{combo_key}"
if key in summary['figures']:
    print(f"Showing: {key} - scatter")
    summary['figures'][key]['scatter'].show()
    print(f"Showing: {key} - trajectory")
    summary['figures'][key]['trajectory'].show()
else:
    print(f"Key '{key}' not found. Available: {list(summary['figures'].keys())}")

Showing: SpontFB_Dopamine - scatter


Showing: SpontFB_Dopamine - trajectory


## Single Dataset Exploration

Use `plot_pca()` directly for a single dataset/combo with interactive display.

In [ ]:
result = plot_pca(
    mat_file="dataSpontFB.mat",
    var_name="dataSpontFB",
    dataset_name="SpontFB",
    neuron_groups=["DF", "DB"],
    neuron_combo_label="Dopamine",
    output_dir=None,   # no PNG saving for exploration
    show=True,
)
print(f"Explained variance: {result['explained_variance_ratio']}")
print(f"Total: {sum(result['explained_variance_ratio']):.4f}")
print(f"Neurons used: {result['n_neurons']} ({result['neuron_groups_used']})")
if result['warnings']:
    print(f"Warnings: {result['warnings']}")